# ORIENT'IA — Analyse exploratoire du jeu de données (ML-1)

Livrable 7 du sujet (« les notebooks d'analyse et d'entraînement »), §7
(« une analyse exploratoire des données »).

**Le calcul n'est pas dans ce notebook.** Il vit dans `src/ml/exploration.py`, testé par
`backend/tests/ml/test_exploration.py`. Un notebook n'est pas testable et dépend de
l'ordre d'exécution de ses cellules ; ce qui doit rester vrai est donc du code testé, et
le notebook se contente de l'appeler et d'en commenter les résultats.

**Ce que l'exploration cherche ici.** Le jeu est *synthétique* : l'enjeu n'est pas de
découvrir des faits sur le monde, mais de **vérifier les hypothèses de génération** et de
repérer les fuites. Une variable qui identifierait à elle seule la classe est le défaut le
plus coûteux — il s'en est déjà produit un (section 4).

In [1]:
# Les notebooks vivent dans backend/notebooks/ : on remonte a backend/ pour que `src` soit
# importable (meme racine que `cd backend && python -m tests.eval_ml`, cf. pyproject.toml).
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [2]:
from src.ml.donnees_synthetiques import charger_jeu_de_donnees
from src.ml.exploration import (
    analyser,
    completude_des_champs,
    correlations_traits_classe,
    distribution_des_classes,
    pouvoir_discriminant_par_champ,
    traits_les_plus_frequents,
)

exemples = charger_jeu_de_donnees()
print(f'{len(exemples)} profils charges')

800 profils charges


## 1. Distribution des classes

Le sujet demande de regarder les **déséquilibres** (§7). Ici l'équilibre est parfait *par
construction* : `generer_jeu_de_donnees` tire `n_par_parcours` profils pour chacun des 16
parcours. Ce n'est donc pas une propriété des données, c'est une décision de génération.

Conséquence à garder en tête au moment de lire les métriques : macro-moyenne et
micro-moyenne coïncident, ce qui ne sera **plus vrai** sur les réponses d'enquête réelles,
où l'auto-sélection sur-représentera certains parcours (§5 du sujet).

In [3]:
d = distribution_des_classes(exemples)
print(f"classes={d['nombre_de_classes']}  total={d['effectif_total']}  "
      f"equilibre={d['equilibre']}")
for parcours, n in d['par_classe'].items():
    print(f'  {parcours:10} {n}')

classes=16  total=800  equilibre=True
  AEE        50
  CAA        50
  DTJA       50
  EMII       50
  EMP        50
  ESIIA      50
  FIC        50
  GCA        50
  IAA        50
  ICMP       50
  IGGLIA     50
  IMTICIA    50
  ISAIA      50
  PIP        50
  TEE        50
  TEH        50


## 2. Complétude des champs

Un champ jamais renseigné n'est pas une statistique anodine : c'est une capacité que le
modèle **ne peut pas** apprendre.

In [4]:
for champ, stats in completude_des_champs(exemples).items():
    print(f"  {champ:34} {stats['taux']:.0%}  ({stats['renseigne']} profils)")

  matieres_preferees                 100%  (800 profils)
  competences_declarees              100%  (800 profils)
  centres_interet                    100%  (800 profils)
  preferences_professionnelles       100%  (800 profils)
  resultats_scolaires                100%  (800 profils)
  environnement_travail_recherche    100%  (800 profils)
  activites_projets                  0%  (0 profils)
  serie_bac                          0%  (0 profils)


### Deux trous mesurés, à nommer plutôt qu'à masquer

- **`activites_projets` : 0 %.** Le §5 du sujet le liste explicitement parmi les éléments
  d'un profil (« les activités ou projets déjà réalisés »). Le générateur ne le produit
  pas et `features.vectoriser()` ne l'encode pas : l'information est absente de bout en
  bout. Un candidat réel qui la déclare ne verra pas sa déclaration peser sur le score.
- **`serie_bac` : 0 %.** Conséquence directe : les règles d'admission du volet hybride
  (`ml/hybride.py`) sont **inertes sur ce jeu**, puisqu'elles ne s'appliquent qu'à un
  profil déclarant une série. Leur effet est réel mais démontré ailleurs
  (`backend/tests/ml/test_hybride.py`), pas mesurable ici.

Aucun des deux n'est un défaut du modèle : ce sont des limites du jeu synthétique, qui
disparaîtront avec l'enquête réelle (DATA-4, ML-7).

## 3. Traits les plus fréquents

In [5]:
for champ, traits in traits_les_plus_frequents(exemples, top_n=6).items():
    print(f'\n{champ} :')
    for t in traits:
        print(f"   {t['trait']:32} {t['occurrences']}")


matieres_preferees :
   mathematiques                    229
   informatique                     223
   economie                         160
   biologie                         148
   chimie                           122
   physique                         108

competences_declarees :
   chimie_industrielle              96
   gestion_de_projet                93
   programmation                    74
   marketing                        60
   gestion_hoteliere                60
   analyse_financiere               57

centres_interet :
   finance                          85
   entrepreneuriat                  77
   voyage                           75
   donnees                          58
   hospitalite                      58
   environnement_rural              57

preferences_professionnelles :
   communication_digitale           62
   recherche_pharmaceutique         61
   data_analyst                     58
   maintenance                      57
   industrie_pharma                 57

## 4. Recherche de fuite — le contrôle le plus important de ce notebook

**Contexte.** Une première version du générateur laissait `environnement_travail_recherche`
fixe et unique par archétype. Avec cette seule fuite, n'importe quel modèle atteignait
100 % d'exactitude sur 16 classes, quel que soit le bruit ajouté ailleurs : la variable
suffisait à identifier le parcours. Le défaut a été trouvé, mesuré, corrigé et documenté
(`donnees_synthetiques.py`).

**Ce contrôle existe pour que le prochain se voie**, sans avoir à ré-entraîner quoi que ce
soit : pour chaque champ, quelle part des traits n'apparaît que dans **une seule** classe ?
Une part proche de 1 signalerait que le modèle n'a qu'à lire ce champ.

In [6]:
for champ, stats in pouvoir_discriminant_par_champ(exemples).items():
    print(f"  {champ:34} exclusifs a 1 classe : "
          f"{stats['part_traits_exclusifs']:.0%}"
          f"   ({stats['classes_moyennes_par_trait']:.1f} classes par trait)")

  matieres_preferees                 exclusifs a 1 classe : 0%   (13.3 classes par trait)
  competences_declarees              exclusifs a 1 classe : 0%   (13.2 classes par trait)
  centres_interet                    exclusifs a 1 classe : 0%   (12.2 classes par trait)
  preferences_professionnelles       exclusifs a 1 classe : 0%   (13.3 classes par trait)
  environnement_travail_recherche    exclusifs a 1 classe : 0%   (10.3 classes par trait)


**Lecture.** 0 % de traits exclusifs sur tous les champs, et chaque trait apparaît en
moyenne dans une dizaine des 16 classes : aucune variable ne sépare seule les parcours.
C'est le résultat attendu après correction, et il est désormais **mesuré** à chaque
exécution plutôt qu'affirmé.

## 5. Quels traits portent le signal

Corrélation point-bisériale entre chaque dimension du vecteur et l'appartenance à une
classe, calculée sur l'espace de features **réellement utilisé par le modèle**
(`noms_features()`) et non sur les champs bruts.

In [7]:
for c in correlations_traits_classe(exemples, top_n=15):
    print(f"  {c['parcours']:10} {c['trait']:44} r={c['correlation']:+.3f}")

  CAA        environnement:bureau_terrain_commercial      r=+0.770
  IMTICIA    environnement:studio_creatif                 r=+0.770
  IAA        environnement:usine_agroalimentaire          r=+0.757
  EMII       environnement:usine_atelier                  r=+0.728
  PIP        environnement:laboratoire                    r=+0.720
  TEH        environnement:hotel_contact_client           r=+0.718
  ESIIA      environnement:laboratoire_atelier            r=+0.715
  DTJA       environnement:bureau_cabinet                 r=+0.701
  AEE        environnement:terrain_rural                  r=+0.698
  IGGLIA     environnement:bureau_informatique            r=+0.690
  TEE        environnement:terrain_nature                 r=+0.658
  DTJA       competence:redaction                         r=+0.654
  ISAIA      environnement:bureau_analytique              r=+0.654
  ICMP       environnement:site_industriel_terrain        r=+0.648
  GCA        environnement:chantier_bureau_etudes         r=+0

**Lecture.** Les corrélations les plus fortes relient un parcours aux traits de son propre
archétype — c'est la vérification que la génération fait ce qu'elle annonce. Leur
amplitude modérée (et non proche de 1) confirme que le bruit croisé joue son rôle : le
lien existe sans être déterministe.

**Limite à garder en tête.** Ces corrélations décrivent les *hypothèses d'archétypes*, pas
un lien réel entre profil et parcours. Un modèle qui les retrouve prouve qu'il apprend le
générateur, pas qu'il sait orienter (§5 du sujet).

## 6. Analyse complète, sérialisable

In [8]:
import json

rapport = analyser(exemples)
print(json.dumps(rapport['distribution_des_classes'], indent=2,
                 ensure_ascii=False)[:400])
print(f"\ndimension de l'espace de features : "
      f"{rapport['dimension_de_l_espace_de_features']}")

{
  "nombre_de_classes": 16,
  "effectif_total": 800,
  "par_classe": {
    "AEE": 50,
    "CAA": 50,
    "DTJA": 50,
    "EMII": 50,
    "EMP": 50,
    "ESIIA": 50,
    "FIC": 50,
    "GCA": 50,
    "IAA": 50,
    "ICMP": 50,
    "IGGLIA": 50,
    "IMTICIA": 50,
    "ISAIA": 50,
    "PIP": 50,
    "TEE": 50,
    "TEH": 50
  },
  "effectif_min": 50,
  "effectif_max": 50,
  "equilibre": true
}

dimension de l'espace de features : 156
